# Task 2: Exploratory Data Analysis (EDA)

**CodSoft Internship** — Dataset: Superstore Sales (cleaned)

## 1. Load Cleaned Dataset

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Load the cleaned dataset from Task 1
df = pd.read_csv('../Task-1-Data-Cleaning/data/processed/cleaned_dataset.csv')
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Date range: {df["OrderDate"].min()} to {df["OrderDate"].max()}')


## 2. Descriptive Statistics

In [ ]:

numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
stats = df[numeric_cols].describe().round(2)
print(stats.to_string())
print(f'\nTotal Revenue: ${df["Sales"].sum():,.2f}')
print(f'Total Profit: ${df["Profit"].sum():,.2f}')
print(f'Profit Margin: {df["Profit"].sum()/df["Sales"].sum()*100:.2f}%')


## 3. Distribution Analysis

In [ ]:

print('Category distribution:')
print(df['Category'].value_counts().to_string())
print('\nSegment distribution:')
print(df['Segment'].value_counts().to_string())
print('\nProfit distribution:')
print(f'Positive profit: {len(df[df["Profit"] > 0])} ({len(df[df["Profit"] > 0])/len(df)*100:.1f}%)')
print(f'Negative profit: {len(df[df["Profit"] < 0])} ({len(df[df["Profit"] < 0])/len(df)*100:.1f}%)')


## 4. Trend Analysis

In [ ]:

df['OrderYear'] = df['OrderDate'].dt.year
df['OrderMonth'] = df['OrderDate'].dt.month

# Yearly trends
yearly = df.groupby('OrderYear').agg({'Sales': 'sum', 'Profit': 'sum'})
print('Yearly sales and profit:')
print(yearly.round(2).to_string())

# Monthly seasonality
monthly = df.groupby('OrderMonth')['Sales'].sum()
print(f'\nPeak month: {monthly.idxmax()} (${monthly.max():,.2f})')
print(f'Lowest month: {monthly.idxmin()} (${monthly.min():,.2f})')


## 5. Relationship Analysis (Correlation)

In [ ]:

corr = df[numeric_cols].corr().round(4)
print('Correlation matrix:')
print(corr.to_string())

print(f'\nSales-Profit correlation: {corr.loc["Sales","Profit"]:.4f}')
print(f'Profit-Discount correlation: {corr.loc["Profit","Discount"]:.4f}')

# Category analysis
print('\nSales and profit by category:')
print(df.groupby('Category').agg({'Sales': 'sum', 'Profit': 'sum'}).round(2).to_string())


## 6. Outlier Detection (IQR Method)

In [ ]:

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    low_out = df[df[col] < lower]
    high_out = df[df[col] > upper]
    print(f'{col}: Low={len(low_out)}, High={len(high_out)} outliers')


## 7. Unusual Patterns

In [ ]:

# High discount, loss-making orders
hdl = df[(df['Discount'] >= 0.5) & (df['Profit'] < 0)]
print(f'High-discount loss-making orders: {len(hdl)} ({len(hdl)/len(df)*100:.1f}%)')
print('\nTop loss-making products:')
print(df[df['Profit'] < 0].groupby('ProductName')['Profit'].sum().sort_values().head(5).to_string())


## 8. Business Questions

In [ ]:

# Q1: Most revenue category
rev_by_cat = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print(f'Q1: Top revenue category: {rev_by_cat.index[0]} (${rev_by_cat.iloc[0]:,.2f})')

# Q2: Most profitable segment
prof_by_seg = df.groupby('Segment')['Profit'].sum().sort_values(ascending=False)
print(f'Q2: Most profitable segment: {prof_by_seg.index[0]} (${prof_by_seg.iloc[0]:,.2f})')

# Q3: Highest sales region
sales_by_reg = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
print(f'Q3: Highest sales region: {sales_by_reg.index[0]} (${sales_by_reg.iloc[0]:,.2f})')

# Q4: Overall profitability
print(f'Q4: Revenue=${df["Sales"].sum():,.2f}, Profit=${df["Profit"].sum():,.2f}')
